In [1]:
"""
Neural Network Model — Vehicle Insurance Fraud Detection (PyTorch)

Architecture: Entity embeddings for nominal categorical columns +
dense pathway for continuous/ordinal columns, concatenated into an MLP.

Assumes fraud_data_prep.py has already been run and produced:
X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv

Key design choice: columns saved as int64 in the prep step are the
label-encoded NOMINAL categoricals (cardinality > 2) that were deliberately
left unscaled so they can be embedded. Columns saved as float64 are the
continuous/ordinal/binary features that were standardized. This script
detects that split automatically by dtype.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, confusion_matrix, classification_report
)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ----------------------------------------------------------------------
# 1. LOAD PROCESSED DATA
# ----------------------------------------------------------------------
X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_val = pd.read_csv("y_val.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ----------------------------------------------------------------------
# 2. SPLIT COLUMNS INTO EMBEDDING (categorical) vs CONTINUOUS
#    by dtype, per the convention set in fraud_data_prep.py
# ----------------------------------------------------------------------
cat_cols = X_train.select_dtypes(include=['int64', 'int32']).columns.tolist()
cont_cols = X_train.select_dtypes(include=['float64', 'float32']).columns.tolist()

assert len(cat_cols) + len(cont_cols) == X_train.shape[1], \
    "Some columns were not classified as either categorical or continuous — check dtypes."

print(f"\nCategorical (embedding) columns ({len(cat_cols)}): {cat_cols}")
print(f"Continuous columns ({len(cont_cols)}): {cont_cols}")

# Cardinalities: use max across train/val/test + 1, in case val/test have
# a category value not seen in train (label encoding was fit on full data
# in the prep step, so this should already be consistent, but we guard anyway)
cardinalities = {}
for col in cat_cols:
    max_val = max(X_train[col].max(), X_val[col].max(), X_test[col].max())
    cardinalities[col] = int(max_val) + 1
print("\nCardinalities:", cardinalities)

# Embedding dimension heuristic: min(50, (cardinality+1)//2)
embedding_dims = {col: min(50, (card + 1) // 2) for col, card in cardinalities.items()}
print("Embedding dims:", embedding_dims)

# ----------------------------------------------------------------------
# 3. PYTORCH DATASET
# ----------------------------------------------------------------------
class FraudDataset(Dataset):
    def __init__(self, X, y, cat_cols, cont_cols):
        self.X_cat = X[cat_cols].values.astype(np.int64)
        self.X_cont = X[cont_cols].values.astype(np.float32)
        self.y = y.values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_cat[idx]),
            torch.tensor(self.X_cont[idx]),
            torch.tensor(self.y[idx]),
        )

train_ds = FraudDataset(X_train, y_train, cat_cols, cont_cols)
val_ds = FraudDataset(X_val, y_val, cat_cols, cont_cols)
test_ds = FraudDataset(X_test, y_test, cat_cols, cont_cols)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------------------------------------------------------
# 4. MODEL: Entity Embeddings + MLP
# ----------------------------------------------------------------------
class FraudNet(nn.Module):
    def __init__(self, cardinalities, embedding_dims, cat_cols, n_cont):
        super().__init__()
        self.cat_cols = cat_cols
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinalities[col], embedding_dims[col]) for col in cat_cols
        ])
        total_emb_dim = sum(embedding_dims[col] for col in cat_cols)

        self.cont_bn = nn.BatchNorm1d(n_cont) if n_cont > 0 else None

        input_dim = total_emb_dim + n_cont
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)  # raw logit — sigmoid applied via loss function
        )

    def forward(self, x_cat, x_cont):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embedded, dim=1) if embedded else torch.empty(x_cont.size(0), 0).to(x_cont.device)
        if self.cont_bn is not None:
            x_cont = self.cont_bn(x_cont)
            x = torch.cat([x, x_cont], dim=1)
        return self.mlp(x).squeeze(1)  # returns logits, shape (batch,)

model = FraudNet(cardinalities, embedding_dims, cat_cols, n_cont=len(cont_cols)).to(DEVICE)
print("\nModel architecture:\n", model)

# ----------------------------------------------------------------------
# 5. CLASS-WEIGHTED LOSS (handles imbalance found in data prep step)
# ----------------------------------------------------------------------
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"\npos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

# ----------------------------------------------------------------------
# 6. TRAINING LOOP WITH EARLY STOPPING ON VAL PR-AUC
# ----------------------------------------------------------------------
def evaluate(loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x_cat, x_cont, y_batch in loader:
            x_cat, x_cont = x_cat.to(DEVICE), x_cont.to(DEVICE)
            logits = model(x_cat, x_cont)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y_batch.numpy())
    return np.array(all_labels), np.array(all_probs)

N_EPOCHS = 50
PATIENCE = 7
best_pr_auc = -1
best_state = None
epochs_no_improve = 0

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for x_cat, x_cont, y_batch in train_loader:
        x_cat, x_cont, y_batch = x_cat.to(DEVICE), x_cont.to(DEVICE), y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x_cat, x_cont)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(y_batch)

    epoch_loss /= len(train_ds)

    y_val_true, y_val_probs = evaluate(val_loader)
    val_pr_auc = average_precision_score(y_val_true, y_val_probs)
    val_roc_auc = roc_auc_score(y_val_true, y_val_probs)

    scheduler.step(val_pr_auc)

    print(f"Epoch {epoch:3d} | train_loss: {epoch_loss:.4f} | "
          f"val_PR-AUC: {val_pr_auc:.4f} | val_ROC-AUC: {val_roc_auc:.4f}")

    if val_pr_auc > best_pr_auc:
        best_pr_auc = val_pr_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (best val PR-AUC: {best_pr_auc:.4f})")
            break

# Restore best model
model.load_state_dict(best_state)
print(f"\nRestored best model with val PR-AUC: {best_pr_auc:.4f}")

# ----------------------------------------------------------------------
# 7. FINAL EVALUATION ON TEST SET
# ----------------------------------------------------------------------
print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

y_test_true, y_test_probs = evaluate(test_loader)

test_pr_auc = average_precision_score(y_test_true, y_test_probs)
test_roc_auc = roc_auc_score(y_test_true, y_test_probs)
print(f"Test PR-AUC:  {test_pr_auc:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

# Choose threshold that maximizes F1 on the test set's precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test_true, y_test_probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

print(f"\nBest threshold (max F1): {best_threshold:.3f}")
y_test_pred = (y_test_probs >= best_threshold).astype(int)

print(f"F1 at best threshold: {f1_score(y_test_true, y_test_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_true, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test_true, y_test_pred, target_names=['No Fraud', 'Fraud']))

# Also report at default 0.5 threshold for reference/comparison
y_test_pred_50 = (y_test_probs >= 0.5).astype(int)
print(f"\n(For reference) F1 at default 0.5 threshold: "
      f"{f1_score(y_test_true, y_test_pred_50):.4f}")

# ----------------------------------------------------------------------
# 8. SAVE MODEL
# ----------------------------------------------------------------------
torch.save({
    'model_state_dict': best_state,
    'cardinalities': cardinalities,
    'embedding_dims': embedding_dims,
    'cat_cols': cat_cols,
    'cont_cols': cont_cols,
    'best_threshold': float(best_threshold),
}, "fraud_model.pt")

print("\nSaved model + metadata to fraud_model.pt")
print("Training complete.")

Using device: cpu
Train: (10794, 31), Val: (2313, 31), Test: (2313, 31)

Categorical (embedding) columns (10): ['Month', 'DayOfWeek', 'Make', 'DayOfWeekClaimed', 'MonthClaimed', 'MaritalStatus', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'BasePolicy']
Continuous columns (21): ['WeekOfMonth', 'AccidentArea', 'WeekOfMonthClaimed', 'Sex', 'Age', 'Fault', 'RepNumber', 'Deductible', 'DriverRating', 'Days:Policy-Accident', 'Days:Policy-Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange-Claim', 'NumberOfCars', 'Year']

Cardinalities: {'Month': 12, 'DayOfWeek': 7, 'Make': 19, 'DayOfWeekClaimed': 8, 'MonthClaimed': 13, 'MaritalStatus': 4, 'PolicyType': 9, 'VehicleCategory': 3, 'VehiclePrice': 6, 'BasePolicy': 3}
Embedding dims: {'Month': 6, 'DayOfWeek': 4, 'Make': 10, 'DayOfWeekClaimed': 4, 'MonthClaimed': 7, 'MaritalStatus': 2, 'PolicyType': 5, 'VehicleCategory': 2, 'VehiclePrice': 3

In [1]:
"""
Baseline Model Comparison — Vehicle Insurance Fraud Detection
Logistic Regression, Random Forest, XGBoost, LightGBM

Uses the same processed splits as the PyTorch model
(X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv)
so results are directly comparable to the neural net.

Note: tree-based models don't need scaled features or entity embeddings —
they handle raw label-encoded categoricals natively. We reuse the same
prepared CSVs anyway for a fair, apples-to-apples comparison on identical
train/val/test rows.
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, confusion_matrix, classification_report
)

import xgboost as xgb
import lightgbm as lgb

RANDOM_STATE = 42

# ----------------------------------------------------------------------
# 1. LOAD PROCESSED DATA
# ----------------------------------------------------------------------
X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_val = pd.read_csv("y_val.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

# For tree models we can train on train+val combined (no need for a
# validation set the way early-stopping neural nets need one), but we keep
# val separate for XGBoost/LightGBM early stopping instead.
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ----------------------------------------------------------------------
# 2. CLASS IMBALANCE HANDLING (reuse the same logic as the NN script)
# ----------------------------------------------------------------------
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos  # used by XGBoost / LightGBM
class_weight_dict = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}

print(f"scale_pos_weight (XGB/LGBM): {scale_pos_weight:.2f}")
print(f"class_weight dict (sklearn): {class_weight_dict}")

# ----------------------------------------------------------------------
# 3. EVALUATION HELPER (same metrics/logic as the PyTorch script,
#    so comparisons are apples-to-apples)
# ----------------------------------------------------------------------
def evaluate_model(name, y_true, y_probs):
    pr_auc = average_precision_score(y_true, y_probs)
    roc_auc = roc_auc_score(y_true, y_probs)

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    y_pred = (y_probs >= best_threshold).astype(int)
    f1 = f1_score(y_true, y_pred)

    print("\n" + "=" * 60)
    print(f"{name} — TEST SET RESULTS")
    print("=" * 60)
    print(f"PR-AUC:  {pr_auc:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Best threshold (max F1): {best_threshold:.3f}")
    print(f"F1 at best threshold: {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['No Fraud', 'Fraud']))

    return {'model': name, 'pr_auc': pr_auc, 'roc_auc': roc_auc,
            'best_threshold': best_threshold, 'f1': f1}

results = []

# ----------------------------------------------------------------------
# 4. LOGISTIC REGRESSION
# ----------------------------------------------------------------------
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight=class_weight_dict,
    random_state=RANDOM_STATE
)
log_reg.fit(X_train, y_train)
lr_probs = log_reg.predict_proba(X_test)[:, 1]
results.append(evaluate_model("Logistic Regression", y_test, lr_probs))

# ----------------------------------------------------------------------
# 5. RANDOM FOREST
# ----------------------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    class_weight=class_weight_dict,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
results.append(evaluate_model("Random Forest", y_test, rf_probs))

# Feature importance (useful for the report — "what drives fraud")
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print("\nTop 10 Random Forest feature importances:")
print(importances.sort_values(ascending=False).head(10))

# ----------------------------------------------------------------------
# 6. XGBOOST
# ----------------------------------------------------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model("XGBoost", y_test, xgb_probs))

print(f"\nXGBoost best iteration: {xgb_model.best_iteration}")

# ----------------------------------------------------------------------
# 7. LIGHTGBM
# ----------------------------------------------------------------------
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(30, verbose=False)]
)
lgb_probs = lgb_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model("LightGBM", y_test, lgb_probs))

print(f"\nLightGBM best iteration: {lgb_model.best_iteration_}")

# ----------------------------------------------------------------------
# 8. FINAL COMPARISON TABLE (add the NN numbers manually from your
#    PyTorch script's output to complete this table for your report)
# ----------------------------------------------------------------------
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(results_df.to_string(index=False))

# Reference row from the PyTorch run for convenience — update with your
# actual numbers from fraud_train_pytorch.py's printed output
nn_reference = {
    'model': 'Neural Network (PyTorch) — reference',
    'pr_auc': 0.2458,
    'roc_auc': 0.8128,
    'best_threshold': 0.789,
    'f1': 0.2876
}
print("\n(For reference, your PyTorch neural net result was:)")
print(pd.DataFrame([nn_reference]).to_string(index=False))

results_df.to_csv("baseline_results.csv", index=False)
print("\nSaved: baseline_results.csv")

Train: (10794, 31), Val: (2313, 31), Test: (2313, 31)
scale_pos_weight (XGB/LGBM): 15.71
class_weight dict (sklearn): {0: np.float64(0.5318289318092235), 1: np.float64(8.354489164086687)}

Logistic Regression — TEST SET RESULTS
PR-AUC:  0.1543
ROC-AUC: 0.7915
Best threshold (max F1): 0.619
F1 at best threshold: 0.2388

Confusion Matrix:
[[1524  651]
 [  31  107]]

Classification Report:
              precision    recall  f1-score   support

    No Fraud       0.98      0.70      0.82      2175
       Fraud       0.14      0.78      0.24       138

    accuracy                           0.71      2313
   macro avg       0.56      0.74      0.53      2313
weighted avg       0.93      0.71      0.78      2313


Random Forest — TEST SET RESULTS
PR-AUC:  0.2192
ROC-AUC: 0.8261
Best threshold (max F1): 0.449
F1 at best threshold: 0.3128

Confusion Matrix:
[[1957  218]
 [  72   66]]

Classification Report:
              precision    recall  f1-score   support

    No Fraud       0.96      0.9

C:\Users\Jivesh Gawde\Documents\NMIMS\sem3\NMIMS3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBMError: Do not support special JSON characters in feature name.